<a href="https://colab.research.google.com/github/Ash100/Protein-Language-Models-Predict-LXXLL-Viral-Motifs-/blob/main/Viral_Protein_Motif_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Viral Protein LXXLL Motif Search

**Part of:** *Protein Language Models Predict Candidate Viral Coactivator-Mimicking Motifs Targeting the
Androgen and Estrogen Receptors: An In Silico Study*

This notebook performs **Stage 1** of the analysis pipeline: retrieving viral protein sequences from NCBI
and identifying candidate **LXXLL** ("NR box") motifs — the short leucine-rich sequence motif used by host
nuclear receptor coactivators (e.g., NCOA1–4, CBP, MED1, PGC1α) to engage the androgen receptor (AR) and
estrogen receptor (ER) ligand-binding domains.

The curated, motif-containing protein set produced here is the direct input to the downstream ESM2
embedding, UMAP/clustering, and structural modelling notebooks in this repository.

## What this notebook does

1. **Fetch** protein records for viruses from the NCBI Protein database via `Biopython`'s
   `Entrez` interface (max 20 records per virus).
2. **Deduplicate** the retrieved sequences (exact sequence matches, and redundant protein names within a
   virus).
3. **Scan** each protein sequence for the regular-expression pattern `L[A-Z]{2}LL` (i.e., **L-X-X-L-L**) and
   record the position(s) and matched motif string(s).
4. **(Optional)** Filter/rank the motif-containing proteins for a small, literature-selected subset of
   high-interest viruses.

## Pipeline position

```
[this notebook]                 →  virus_proteins_with_lxxll_motif.csv
        │
        ▼
25-residue LXXLL-centred window extraction (10aa–LXXLL–10aa)
        │
        ▼
ESM2 (ESM2-t33_650M_UR50D) embedding  →  UMAP projection  →  clustering (Spectral/KMeans/HDBSCAN/etc.)
        │
        ▼
Candidate filtering (cosine similarity, silhouette score, cluster-centroid proximity)
        │
        ▼
Boltz-2 structural modelling vs. AR/ER LBDs  →  MM-GBSA binding free-energy ranking
```

## Requirements

- Python 3.9+
- `biopython`, `pandas`, `tqdm`, `requests`, `matplotlib` (installed in the first code cell)
- A valid email address registered with `Entrez.email` (required by NCBI for API access — **replace the
  placeholder email in the cells below with your own before running this notebook**)
- Internet access to `https://eutils.ncbi.nlm.nih.gov`

## Inputs / Outputs

| Cell | Output file | Description |
|---|---|---|
| Protein retrieval | `virus_protein_sequences.csv` | Raw fetched sequences: `ID`, `Sequence`, `Virus Name`, `Protein Name` |
| Redundancy filtering | `filtered_virus_protein_sequences.csv` | Deduplicated sequences |
| Motif search | `virus_proteins_with_lxxll_motif.csv` | Proteins with ≥1 LXXLL motif, positions and motif sequences |
| Prioritization (optional) | `prioritized_virus_proteins_lxxll.csv` | Subset ranked by motif count for a small set of high-interest viruses |

> **Note on reproducibility:** NCBI Entrez searches are retrieved dynamically and can return slightly
> different results over time as RefSeq/GenBank records are added, updated, or withdrawn. The exact motif
> dataset used in the published analysis (355 curated 25-residue windows) is provided as a static file in
> [`data/motif_dataset.csv`](../data/motif_dataset.csv) of this repository for exact reproducibility of the
> downstream results, independent of live re-querying of NCBI.

> **Search scope:** The retrieval below requests a maximum of 20 protein records per virus (`retmax=20`) for
> computational tractability and to respect NCBI rate limits. This represents a curated subset rather than
> each virus's complete proteome; viruses with more than 20 annotated protein records may have additional
> LXXLL-motif-containing proteins not captured here.


In [ ]:
!pip -q install biopython pandas tqdm requests matplotlib

### Setup

Install dependencies and import required libraries. `Entrez.email` must be set to a valid email address
before making any NCBI E-utilities requests (this is an NCBI usage requirement, not optional).


In [ ]:
import re
import io
import os
import sys
import json
import time
import gzip
import math
import textwrap
import requests
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from Bio import Entrez, SeqIO

# Always set your email for NCBI Entrez
Entrez.email = "ashfaqahmad82@hotmail.com"


### Step 1 — Retrieve viral protein sequences from NCBI

Queries the NCBI Protein database for each virus in the `viruses` list using an `[Organism]`-restricted
Entrez search, retrieves up to 20 protein records per virus in FASTA format, and parses them into a tabular
dataset (`virus_protein_sequences.csv`) with columns `ID`, `Sequence`, `Virus Name`, `Protein Name`.

A short delay (`time.sleep(0.5)`) is included between requests to stay within NCBI's unauthenticated rate
limit (3 requests/second). If you have an NCBI API key, you can register it with `Entrez.api_key` to
increase this limit.

**Before running:** replace `Entrez.email` below with your own email address.


In [ ]:
#More Specialized code
import os
from Bio import Entrez, SeqIO
import pandas as pd
import time
import re

# Replace with your actual email (NCBI requires this for Entrez access)
Entrez.email = "ashfaqahmad82@hotmail.com"

# List of virus names
viruses = [
    "Herpes Simplex Virus 1",
    "Herpes Simplex Virus 2",
    "Varicella-Zoster Virus",
    "Epstein-Barr Virus",
    "Cytomegalovirus",
    "Human Herpesvirus 6",
    "Human Herpesvirus 7",
    "Kaposi’s Sarcoma-Associated Herpesvirus",
    "Variola Virus",
    "Monkeypox Virus",
    "Molluscum Contagiosum Virus",
    "Cowpox Virus",
    "Human Papillomavirus",
    "BK Virus",
    "JC Virus",
    "Merkel Cell Polyomavirus",
    "Human Adenovirus",
    "Parvovirus B19",
    "Hepatitis B Virus",
    "Influenza A Virus",
    "Influenza B Virus",
    "Influenza C Virus",
    "Influenza D Virus",
    "SARS-CoV",
    "MERS-CoV",
    "SARS-CoV-2",
    "Human Coronavirus 229E",
    "Human Coronavirus NL63",
    "Human Coronavirus OC43",
    "Human Coronavirus HKU1",
    "Poliovirus",
    "Coxsackievirus A",
    "Coxsackievirus B",
    "Echovirus",
    "Enterovirus",
    "Rhinovirus",
    "Hepatitis A Virus",
    "Dengue Virus",
    "Zika Virus",
    "West Nile Virus",
    "Yellow Fever Virus",
    "Hepatitis C Virus",
    "Japanese Encephalitis Virus",
    "Tick-Borne Encephalitis Virus",
    "Chikungunya Virus",
    "Rubella Virus",
    "Ross River Virus",
    "Rabies Virus",
    "Measles Virus",
    "Mumps Virus",
    "Respiratory Syncytial Virus",
    "Human Metapneumovirus",
    "Parainfluenza Virus",
    "Hantavirus",
    "Hantaan Virus",
    "Sin Nombre Virus",
    "Crimean-Congo Hemorrhagic Fever Virus",
    "Lassa Virus",
    "Lymphocytic Choriomeningitis Virus",
    "Ebola Virus",
    "Marburg Virus",
    "Rotavirus",
    "Colorado Tick Fever Virus",
    "Norovirus",
    "Sapovirus",
    "Human Astrovirus",
    "Hepatitis E Virus",
    "Human Immunodeficiency Virus 1",
    "Human Immunodeficiency Virus 2",
    "Human T-Lymphotropic Virus 1",
    "Human T-Lymphotropic Virus 2",
    "Nipah Virus",
    "Hendra Virus",
    "Rift Valley Fever Virus"
]

# Initialize list to store data
data = []

for virus in viruses:
    print(f"Fetching proteins for {virus}")
    try:
        # Search for protein sequences in NCBI Protein database
        # Limit to 10 sequences per virus to avoid overwhelming downloads (adjust retmax as needed)
        handle = Entrez.esearch(db="protein", term=f"{virus}[Organism]", retmax=20)
        record = Entrez.read(handle)
        handle.close()

        id_list = record["IdList"]

        if not id_list:
            print(f"No proteins found for {virus}")
            continue

        # Fetch the sequences in FASTA format
        fetch_handle = Entrez.efetch(db="protein", id=",".join(id_list), rettype="fasta", retmode="text")
        fasta_data = fetch_handle.read()
        fetch_handle.close()

        # Parse FASTA data
        from io import StringIO
        fasta_io = StringIO(fasta_data)
        for record in SeqIO.parse(fasta_io, "fasta"):
            # Extract ID and sequence
            seq_id = record.id
            sequence = str(record.seq)

            # Extract protein name from description (before first '[' or '|')
            description = record.description
            protein_name = re.split(r'\[|\|', description)[0].replace(f">{seq_id} ", "").strip()
            if not protein_name:
                protein_name = "Unknown"

            # Append to data list
            data.append({
                "ID": seq_id,
                "Sequence": sequence,
                "Virus Name": virus,
                "Protein Name": protein_name
            })

        print(f"Processed {len(id_list)} sequences for {virus}")

        # Sleep to respect NCBI rate limits (max 3 requests per second without API key)
        time.sleep(0.5)

    except Exception as e:
        print(f"Error fetching for {virus}: {e}")
        time.sleep(1)  # Longer sleep on error

# Create DataFrame and save to CSV
df = pd.DataFrame(data)
output_file = "virus_protein_sequences.csv"
df.to_csv(output_file, index=False)
print(f"Data saved to {output_file}")

### Step 2 — Remove redundant sequences

Loads `virus_protein_sequences.csv` and removes (a) exact duplicate sequences and (b) duplicate
Virus Name + Protein Name pairs, keeping the first occurrence of each. This reduces redundancy introduced by
multiple RefSeq entries for the same or highly similar protein. Output: `filtered_virus_protein_sequences.csv`.


In [ ]:
#Redundancy Filteration
import pandas as pd

# Load the CSV file generated from the previous step
input_file = "virus_protein_sequences.csv"
output_file = "filtered_virus_protein_sequences.csv"

try:
    # Read the CSV file
    df = pd.read_csv(input_file)

    # Check if the required columns exist
    required_columns = ["ID", "Sequence", "Virus Name", "Protein Name"]
    if not all(col in df.columns for col in required_columns):
        raise ValueError("Input CSV must contain columns: ID, Sequence, Virus Name, Protein Name")

    print(f"Original dataset size: {len(df)} sequences")

    # Remove duplicates based on exact sequence matches
    df = df.drop_duplicates(subset=["Sequence"], keep="first")
    print(f"Dataset size after removing redundant sequences: {len(df)} sequences")

    # Remove duplicates based on Protein Name within each Virus Name, keeping the first occurrence
    df = df.drop_duplicates(subset=["Virus Name", "Protein Name"], keep="first")
    print(f"Dataset size after removing redundant protein names: {len(df)} sequences")

    # Save the filtered dataset to a new CSV
    df.to_csv(output_file, index=False)
    print(f"Filtered dataset saved to {output_file}")

except FileNotFoundError:
    print(f"Error: {input_file} not found. Please ensure the input CSV exists.")
except Exception as e:
    print(f"Error during processing: {e}")

### Step 3a — LXXLL motif search (long format: one row per motif hit)

Scans each filtered protein sequence for the regular expression `L[A-Z]{2}LL` and writes **one row per
motif occurrence** (a protein with multiple motifs will appear on multiple rows), including the 1-indexed
match position and the matched 5-residue motif string.

> This long-format output (`virus_proteins_with_lxxll_motif.csv`) is superseded by the consolidated,
> per-protein version produced in the next cell (Step 3b), which writes to the same filename. Both are kept
> here for transparency into the intermediate motif-detection logic; only Step 3b's output was carried
> forward into the downstream embedding/clustering pipeline.


In [ ]:
import pandas as pd
import re

# Input file from previous step
input_file = "filtered_virus_protein_sequences.csv"
output_file = "virus_proteins_with_lxxll_motif.csv"

# Load the filtered dataset
try:
    df = pd.read_csv(input_file)

    # Check required columns
    required_columns = ["ID", "Sequence", "Virus Name", "Protein Name"]
    if not all(col in df.columns for col in required_columns):
        raise ValueError("Input CSV must contain columns: ID, Sequence, Virus Name, Protein Name")

    print(f"Processing {len(df)} sequences for LXXLL motif...")

    # List to store hits
    hits = []

    # Regex pattern for LXXLL where X is any amino acid (A-Z)
    pattern = re.compile(r'L[A-Z]{2}LL')

    for _, row in df.iterrows():
        sequence = row["Sequence"]
        virus = row["Virus Name"]
        protein_name = row["Protein Name"]
        seq_id = row["ID"]

        # Find all matches with positions (1-indexed)
        for match in pattern.finditer(sequence):
            start_pos = match.start() + 1  # 1-indexed
            motif_seq = match.group()
            hits.append({
                "ID": seq_id,
                "Sequence": sequence,  # Include full sequence
                "Virus Name": virus,
                "Protein Name": protein_name,
                "Motif Position": start_pos,
                "Motif Sequence": motif_seq
            })

    # Create DataFrame for hits
    hits_df = pd.DataFrame(hits)

    if hits_df.empty:
        print("No LXXLL motifs found in any sequences.")
    else:
        # Save to CSV with all columns
        hits_df.to_csv(output_file, index=False)
        print(f"Found {len(hits_df)} motif hits across {hits_df['ID'].nunique()} unique proteins.")
        print(f"Results saved to {output_file}")

        # Optional: Print a preview of hits
        print("\nPreview of hits:")
        # Truncate Sequence column for preview to avoid clutter
        preview_df = hits_df.copy()
        preview_df["Sequence"] = preview_df["Sequence"].apply(lambda x: x[:50] + "..." if len(x) > 50 else x)
        print(preview_df.head(10))  # Show first 10 hits

except FileNotFoundError:
    print(f"Error: {input_file} not found. Please ensure the filtered CSV exists.")
except Exception as e:
    print(f"Error during processing: {e}")

### Step 3b — LXXLL motif search (consolidated format: one row per protein)

Re-scans the filtered protein sequences for the same `L[A-Z]{2}LL` pattern, but consolidates **all** motif
positions and matched sequences for a given protein into a single row (comma-separated `Motif Positions` and
`Motif Sequences` columns). This consolidated file (`virus_proteins_with_lxxll_motif.csv`, overwriting the
Step 3a output above) is the version used for downstream 25-residue window extraction and ESM2 embedding.


In [ ]:
import pandas as pd
import re

# Input file from previous step
input_file = "filtered_virus_protein_sequences.csv"
output_file = "virus_proteins_with_lxxll_motif.csv"

# Load the filtered dataset
try:
    df = pd.read_csv(input_file)

    # Check required columns
    required_columns = ["ID", "Sequence", "Virus Name", "Protein Name"]
    if not all(col in df.columns for col in required_columns):
        raise ValueError("Input CSV must contain columns: ID, Sequence, Virus Name, Protein Name")

    print(f"Processing {len(df)} sequences for LXXLL motif...")

    # Dictionary to store hits
    hits_dict = {}

    # Regex pattern for LXXLL where X is any amino acid (A-Z)
    pattern = re.compile(r'L[A-Z]{2}LL')

    for _, row in df.iterrows():
        sequence = row["Sequence"]
        virus = row["Virus Name"]
        protein_name = row["Protein Name"]
        seq_id = row["ID"]

        # Find all matches with positions (1-indexed)
        matches = pattern.finditer(sequence)
        positions = []
        motifs = []
        for match in matches:
            start_pos = match.start() + 1  # 1-indexed
            motif_seq = match.group()
            positions.append(str(start_pos))
            motifs.append(motif_seq)

        if positions:  # Only add if motifs were found
            hits_dict[seq_id] = {
                "ID": seq_id,
                "Sequence": sequence,
                "Virus Name": virus,
                "Protein Name": protein_name,
                "Motif Positions": ",".join(positions),
                "Motif Sequences": ",".join(motifs)
            }

    # Create DataFrame from hits
    hits_df = pd.DataFrame(list(hits_dict.values()))

    if hits_df.empty:
        print("No LXXLL motifs found in any sequences.")
    else:
        # Save to CSV with all columns
        hits_df.to_csv(output_file, index=False)
        print(f"Found {len(hits_df)} proteins with LXXLL motifs.")
        print(f"Results saved to {output_file}")

        # Optional: Print a preview of hits
        print("\nPreview of hits:")
        # Truncate Sequence column for preview to avoid clutter
        preview_df = hits_df.copy()
        preview_df["Sequence"] = preview_df["Sequence"].apply(lambda x: x[:50] + "..." if len(x) > 50 else x)
        print(preview_df.head(10))  # Show first 10 hits

except FileNotFoundError:
    print(f"Error: {input_file} not found. Please ensure the filtered CSV exists.")
except Exception as e:
    print(f"Error during processing: {e}")

### Step 4 (optional/exploratory) — Prioritize a literature-selected subset of viruses

Filters the consolidated motif dataset down to a small, literature-selected list of viruses considered
high-interest a priori (e.g., SARS-CoV-2, HIV-1/2, KSHV, HBV) and ranks the resulting proteins by motif
count. This step was used for early exploratory review of high-profile candidates and is **not** the
selection criterion used for the final prioritized candidate list reported in the manuscript, which draws
from the full 53-virus, embedding/clustering-based analysis (see downstream notebooks). It is retained here
for transparency of the exploratory analysis process.


In [ ]:
import pandas as pd

# Input CSV from the consolidated motif script
input_file = "virus_proteins_with_lxxll_motif.csv"
output_file = "prioritized_virus_proteins_lxxll.csv"

# High-potential viruses based on literature
priority_viruses = [
    "SARS-CoV-2",
    "Human Immunodeficiency Virus 1",
    "Human Immunodeficiency Virus 2",
    "Kaposi’s Sarcoma-Associated Herpesvirus",
    "Hepatitis B Virus"
]

try:
    df = pd.read_csv(input_file)

    # Check required columns
    required_columns = ["ID", "Sequence", "Virus Name", "Protein Name", "Motif Positions", "Motif Sequences"]
    if not all(col in df.columns for col in required_columns):
        raise ValueError("Input CSV must contain columns: ID, Sequence, Virus Name, Protein Name, Motif Positions, Motif Sequences")

    # Filter to priority viruses
    filtered_df = df[df["Virus Name"].isin(priority_viruses)]

    if filtered_df.empty:
        print("No proteins found for priority viruses. Consider expanding the list.")
    else:
        # Calculate motif count from comma-separated Motif Positions
        filtered_df["Motif Count"] = filtered_df["Motif Positions"].apply(lambda x: len(x.split(",")) if pd.notna(x) else 0)

        # Sort by virus and motif count (descending)
        prioritized_df = filtered_df.sort_values(by=["Virus Name", "Motif Count"], ascending=[True, False])

        # Save to CSV
        prioritized_df.to_csv(output_file, index=False)
        print(f"Prioritized {len(prioritized_df)} proteins saved to {output_file}")
        print("\nPreview:")
        # Truncate Sequence for preview
        preview_df = prioritized_df.copy()
        preview_df["Sequence"] = preview_df["Sequence"].apply(lambda x: x[:50] + "..." if len(x) > 50 else x)
        print(preview_df[["ID", "Virus Name", "Protein Name", "Motif Count", "Motif Positions", "Motif Sequences"]].head(10))

except FileNotFoundError:
    print(f"Error: {input_file} not found.")
except Exception as e:
    print(f"Error: {e}")

---

## Next steps in the pipeline

The output of this notebook (`virus_proteins_with_lxxll_motif.csv`) feeds into:

1. **25-residue window extraction** — each LXXLL motif is expanded to a 25-residue window
   (10 amino acids upstream + LXXLL + 10 amino acids downstream).
2. **ESM2 embedding** (`ESM2-t33_650M_UR50D`) of viral motif windows and host coactivator control motifs
   (NCOA1–4, CBP, MED1, PGC1α).
3. **UMAP projection and clustering** to identify viral motifs occupying the same embedding-space
   neighborhood as known coactivators.
4. **Structural modelling** (Boltz-2 + AlphaFold2 relaxation) and **MM-GBSA** ranking of candidate
   viral–receptor complexes against the AR and ER ligand-binding domains.

See the other notebooks/scripts in this repository for these downstream steps.

## Citation

If you use this notebook or the resulting dataset, please cite:

> [Author list]. *Protein Language Models Predict Candidate Viral Coactivator-Mimicking Motifs Targeting the
> Androgen and Estrogen Receptors: An In Silico Prioritization Study.* [Journal], [year]. DOI: [TBD]

## License

See the repository [`LICENSE`](../LICENSE) file.

## Disclaimer

This notebook performs a purely computational sequence-motif search. The presence of an LXXLL motif does not
by itself imply receptor binding, transcriptional activity, or any other biological function; all candidates
identified here require experimental validation, as described in the associated manuscript.
